# EXPERIMENT 5: DIFFERENTIAL PULSE CODE MODULATION (DPCM), DELTA MODULATION (DM), AND ADAPTIVE DELTA MODULATION (ADM)
### Software-Based Digital Communication Laboratory (EC593 / EC592)
**Institution:** Cooch Behar Government Engineering College, Department of Electronics & Communication Engineering  
**Presenter / Student Name:** Apurba Maity | **University Roll No.:** 34900324001 | **Semester:** 5th Sem ECE  

---

## 1. Laboratory Objectives
1. **Redundancy Reduction in Correlated Sources:** Investigate why Nyquist-sampled speech and analog signals have high correlation ($\rho \approx 0.85 - 0.98$) and how differential encoding eliminates redundant bits.
2. **DPCM Implementation:** Construct a complete DPCM encoder and decoder from first principles in Python (NumPy) using 1st- and 2nd-order optimal linear predictors.
3. **Prediction Gain ($G_p$) Verification:** Mathematically prove and empirically demonstrate the Prediction Gain $G_p = \frac{\sigma_x^2}{\sigma_d^2} \approx 7.2\text{ dB}$, showing how it achieves a $1.2$-bit/sample reduction over standard PCM for equivalent fidelity.
4. **Linear Delta Modulation (DM):** Build a 1-bit quantizer with a local feedback accumulator to evaluate tracking slew rate.
5. **Dual Distortion Analysis:** Empirically demonstrate and mathematically isolate the two limiting regimes of Delta Modulation:
   - **Slope Overload Distortion:** When signal velocity exceeds staircase capacity: $|\dot{x}|_{\max} > \Delta \cdot f_s$
   - **Granular (Idle-Channel) Noise:** Triangular hunting oscillations on slow signals: $N_g = \frac{\Delta^2}{3}$
6. **Adaptive Delta Modulation (ADM):** Implement the **Jayant/Song step-size adaptation algorithm** to eliminate both slope overload and granular noise simultaneously.
7. **Reconstruction & Oversampling SNR Scaling:** Analyze 4th-order Butterworth low-pass reconstruction filtering and verify the theoretical $+9\text{ dB/octave}$ SNR scaling with oversampling frequency: $\mathrm{SNR} \propto (f_s)^3$.

---

## 2. Mathematical Formulations & Derivations

### 2.1 The Prediction Residual & Linear Prediction
In physical speech, continuous mechanics prevent sudden discontinuous energy changes. Adjacent samples have high normalized correlation $\rho = R_{xx}(1) / R_{xx}(0)$.

Instead of quantizing $x[n]$ directly, DPCM predicts the current sample from $p$ past reconstructed samples:
$$\hat{x}[n] = \sum_{k=1}^p a_k \tilde{x}[n-k]$$
The prediction error residual is:
$$d[n] = x[n] - \hat{x}[n]$$

#### Wiener-Hopf Optimal Predictor (Order $p=1$):
Minimizing $E[d^2[n]]$ gives $a_1 = \rho$. The resulting prediction residual variance is:
$$\sigma_d^2 = \sigma_x^2 (1 - \rho^2)$$

#### Prediction Gain ($G_p$):
$$G_p = \frac{\sigma_x^2}{\sigma_d^2} = \frac{1}{1 - \rho^2} \implies G_{p,\mathrm{dB}} = 10 \log_{10}\left(\frac{1}{1 - \rho^2}\right)$$
For typical speech ($\rho = 0.90$), $G_p = 5.26 \approx \mathbf{7.21\text{ dB}}$. This yields an immediate savings of $\approx 1.2$ bits per sample compared to PCM.

#### Mathematical Proof of Non-Accumulation of Quantization Errors:
Let $d_q[n] = d[n] + q[n]$. The transmitter local reconstruction is $\tilde{x}[n] = \hat{x}[n] + d_q[n]$. The receiver performs identical reconstruction: $\tilde{x}[n] = \hat{x}[n] + d_q[n]$.
$$e[n] = x[n] - \tilde{x}[n] = x[n] - (\hat{x}[n] + d_q[n]) = (x[n] - \hat{x}[n]) - d_q[n] = d[n] - d_q[n] = -q[n]$$
**Key Finding:** Overall reconstruction error equals precisely the negative of the single-sample quantization error. Quantization errors do **not** accumulate across time.

### 2.2 Linear Delta Modulation: Slew Rate vs. Granular Noise
Linear DM is a 1-bit DPCM quantizer ($L = 2$ levels, $\pm \Delta$).
$$\tilde{x}[n] = \tilde{x}[n-1] + \Delta \cdot \mathrm{sgn}(x[n] - \tilde{x}[n-1])$$

#### Condition to Prevent Slope Overload Distortion:
For $x(t) = A_m \sin(2\pi f_m t)$, the maximum slope is $|\dot{x}(t)|_{\max} = 2\pi f_m A_m$. The maximum tracking slew rate of the staircase is $\frac{\Delta}{T_s} = \Delta \cdot f_s$.
$$\Delta \cdot f_s \ge 2\pi f_m A_m \implies \mathbf{\Delta \ge \frac{2\pi f_m A_m}{f_s}}$$

#### Granular Noise Power:
When $|\dot{x}| \ll \Delta \cdot f_s$, the staircase hunts around the flat signal with peak error $\pm \Delta$. Modeling this error as uniformly distributed over $[-\Delta, +\Delta]$:
$$N_g = \frac{(2\Delta)^2}{12} = \frac{\Delta^2}{3}$$

### 2.3 Adaptive Delta Modulation (ADM): Jayant Step Adaptation
To resolve the conflict between slope overload and granular noise, the Jayant/Song algorithm modulates $\Delta[n]$ dynamically based on 1-bit output history:
$$\Delta[n] = \begin{cases} \min(\Delta[n-1] \cdot K_e, \Delta_{\max}), & \text{if } b[n] == b[n-1] \quad (\text{monotonic slope } \implies \text{expand step by } K_e = 1.5) \\[1.5mm] \max(\Delta[n-1] \cdot K_c, \Delta_{\min}), & \text{if } b[n] \ne b[n-1] \quad (\text{alternating hunting } \implies \text{compress step by } K_c = 0.66) \end{cases}$$

In [1]:
# Import core scientific libraries and configure presentation display formatting
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Image, display

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['figure.dpi'] = 150

PLOTS_DIR = os.path.join(os.getcwd(), 'plots')
print("Interactive Presentation Environment Ready. Directory:", PLOTS_DIR)

## 3. First-Principles Python Architecture: Core Function Walkthrough
In the cell below, we highlight the exact first-principles implementation of our DPCM encoder-decoder feedback loop, linear DM engine, and the Jayant ADM step-size multiplier algorithm.

In [2]:
import experiment_05

# Execute the simulation pipeline to run all mathematical models and render 300 DPI figures
experiment_05.run_simulation_and_generate_plots()

## 4. Empirical Waveform Analysis & Generated Visualizations

### 4.1 Figure 1: DPCM Architecture & Prediction Residual Dynamics
Notice how the unquantized difference signal $d[n] = x[n] - \hat{x}[n]$ in subplot (b) exhibits dramatically reduced amplitude and variance ($G_p = 7.18\text{ dB}$) relative to the raw input $x[n]$ in subplot (a). This allows the 3-bit quantizer in (c) to employ tight step sizes $\Delta_d$, delivering high reconstruction fidelity in (d).

In [3]:
display(Image(filename=os.path.join(PLOTS_DIR, 'fig1_dpcm_architecture_and_prediction_gain.png')))

### 4.2 Figure 2: Linear DM — Slope Overload vs. Granular Noise Dilemma
Here we observe the classic operational dilemma of fixed-step Delta Modulation:
1. **Under-stepped ($\Delta < \Delta_{\text{crit}}$):** Severe slope overload during rapid zero-crossings where the staircase cannot keep pace with the signal velocity.
2. **Critical Step ($\Delta \approx \Delta_{\text{crit}}$):** Optimum equilibrium between slew rate and granular hunting.
3. **Over-stepped ($\Delta > \Delta_{\text{crit}}$):** Excessive granular noise with large amplitude chatter during slow-moving waveform peaks.

In [4]:
display(Image(filename=os.path.join(PLOTS_DIR, 'fig2_dm_slope_overload_vs_granular_noise.png')))

### 4.3 Figure 3: Adaptive Delta Modulation (ADM) Waveform Tracking
The Jayant/Song ADM step-size adaptation algorithm rapidly expands $\Delta[n]$ during steep ramps ($25\text{ V/s}$) where linear DM fails, and contracts $\Delta[n]$ during flat plateaus, eliminating both distortion modes simultaneously.

In [5]:
display(Image(filename=os.path.join(PLOTS_DIR, 'fig3_adm_adaptive_tracking.png')))

### 4.4 Figure 4: DM SNR vs. Oversampling Frequency Scaling
The plot validates the theoretical $9\text{ dB/octave}$ ($30\text{ dB/decade}$) SNR scaling with oversampling frequency $f_s$ when paired with a low-pass reconstruction filter, confirming $\mathrm{SNR} \propto (f_s / f_m)^3$.

In [6]:
display(Image(filename=os.path.join(PLOTS_DIR, 'fig4_dm_snr_vs_sampling_frequency.png')))

### 4.5 Figure 5: Comparative SQNR Performance Across Encoders
Bar chart comparison of Standard PCM, DPCM, Linear DM, and ADM across identical transmission bit rates ($16\text{ kbps}$ to $64\text{ kbps}$). DPCM consistently demonstrates a $+7.5\text{ dB}$ performance advantage over standard PCM.

In [7]:
display(Image(filename=os.path.join(PLOTS_DIR, 'fig5_sqnr_comparison_pcm_dpcm_dm_adm.png')))

### 4.6 Figure 6: Low-Pass Reconstruction Filtering & Spectrum
Comparison of raw staircase spectrum vs. Butterworth low-pass filtered demodulated output, illustrating effective suppression of out-of-band high-frequency quantization noise.

In [8]:
display(Image(filename=os.path.join(PLOTS_DIR, 'fig6_reconstruction_filtering_and_psd.png')))

## 5. Theory vs. Practicality & Hardware Realities (The Presentation Core)

### 5.1 Where Textbook Theory Meets Hardware Reality
When explaining this experiment in an engineering presentation, three fundamental real-world trade-offs must be highlighted:

1. **The Myth of Error Accumulation:**
   - *Theory Concern:* When students first see a feedback loop where each sample is reconstructed from previous samples, they intuitively assume that quantization errors will compound and diverge to infinity.
   - *Practical Reality:* Because the encoder embeds the local reconstruction $\tilde{x}[n] = \hat{x}[n] + d_q[n]$ inside the loop, the prediction error seen by the quantizer exactly matches the reconstruction error at the receiver: $e[n] = x[n] - \tilde{x}[n] = -q[n]$. Quantization error is **reset on every sample**, making DPCM completely stable.

2. **Channel Bit Errors and Error Propagation:**
   - *Theory:* In ideal mathematical channels, DPCM and ADM achieve superior SQNR.
   - *Practical Reality:* If an acoustic channel or RF link suffers a single bit error, standard PCM only corrupts that single sample. In DPCM and DM, however, because the accumulator integrates past bits, a single flipped bit creates a **permanent DC offset** that persists until a leaky integrator discharge or periodic PCM sync frame resets the state.

3. **Linear DM vs. ADM in Real Silicon (Bluetooth CVSD):**
   - *Theory:* ADM requires dynamic multiplication logic.
   - *Practical Reality:* In commercial devices (such as Bluetooth Hands-Free voice headsets running ITU-T Continuously Variable Slope Delta Modulation - CVSD), ADM is implemented using simple 3-bit or 4-bit shift registers and analog RC charging networks, delivering acceptable $64\text{ kbps}$ voice quality at micro-watt power consumption.

### 5.2 Summary Engineering Trade-Off Matrix

| Parameter | Standard PCM | DPCM | Linear Delta Mod (DM) | Adaptive Delta Mod (ADM) |
| :--- | :---: | :---: | :---: | :---: |
| **Bits per Sample** | $n = 3 - 8$ bits | $n = 2 - 6$ bits | **$1$ bit** | **$1$ bit** |
| **Sampling Rate ($f_s$)** | Nyquist ($2 f_m$) | Nyquist ($2 f_m$) | Oversampled ($20 - 100 f_m$) | Oversampled ($20 - 100 f_m$) |
| **Hardware Complexity** | High (Flash/SAR ADC) | Moderate-High (Predictor) | **Extremely Low** (1 comparator) | Low (Logic multiplier) |
| **Dynamic Range** | Fixed by $[-V_{\max}, +V_{\max}]$ | Bounded by $\pm 3.2 \sigma_d$ | Poor (fixed step $\Delta$) | **Excellent** (Adaptive $\Delta[n]$) |
| **Channel Error Immunity** | Single-sample impact | Error propagates | DC shift until leak | DC shift until leak |

---

## 6. Video Presentation Conclusion & Summary
In this experiment, we:
- Verified that exploiting sample-to-sample correlation $\rho \approx 0.90$ via linear prediction yields a **$7.2\text{ dB}$ Prediction Gain** in DPCM.
- Demonstrated that Linear Delta Modulation is fundamentally constrained by the conflicting requirements of slope overload ($\Delta \ge \frac{2\pi f_m A_m}{f_s}$) and granular noise ($N_g = \frac{\Delta^2}{3}$).
- Validated that the **Jayant Adaptive Delta Modulation algorithm** dynamically scales $\Delta[n]$, eliminating slope overload on fast transients while maintaining low granular noise on flat regions.
- Confirmed the theoretical **$+9\text{ dB / octave}$** ($30\text{ dB / decade}$) oversampling SNR scaling law using Butterworth low-pass reconstruction filtering.

**Source Code & Artifacts:** Open source on GitHub (`SecretiveCodeRunner/Digital_Communication_Lab`).